In [1]:
import logging
import pathlib
from argparse import ArgumentParser, RawTextHelpFormatter
from dataclasses import dataclass
from functools import partial
from typing import Callable

import torch
import torchaudio
import sys
sys.path.append('/om4/group/mcdermott/user/imgriff/projects/pytorch_audio/audio/examples/asr/emformer_rnnt/')
from common import MODEL_TYPE_LIBRISPEECH
from torchaudio.pipelines import EMFORMER_RNNT_BASE_LIBRISPEECH
from torchaudio.pipelines import RNNTBundle
import torchaudio.transforms as T

sys.path.append('/om4/group/mcdermott/user/imgriff/projects/End-to-end-ASR-Pytorch/')
from corpus.single_word import SingleWordDataset as Dataset
from src.text import load_text_encoder

import fuzzy 
from tqdm.notebook import tqdm
import pandas as pd

### Pytorch demo format 

In [2]:
@dataclass
class Config:
    dataset: Callable
    bundle: RNNTBundle

In [3]:

_CONFIGS = {
    MODEL_TYPE_LIBRISPEECH: Config(
        partial(torchaudio.datasets.LIBRISPEECH, url="test-clean"),
        EMFORMER_RNNT_BASE_LIBRISPEECH)
}

### Set params 

In [4]:
model_type = 'librispeech'

path = '/om2/user/imgriff/projects/cocktail_party/datasets/GigaSpeech_top_200_words/' # Path to raw LibriSpeech dataset
dev_split = ['GigaSpeech_top_200_words']  # Name of data splits to be used as validation set

mode = 'word'                       # 'character'/'word'/'subword'
vocab_file = '/om4/group/mcdermott/user/imgriff/projects/End-to-end-ASR-Pytorch/tests/sample_data/gigaspeech_words-10k.vocab'


### Load model

In [8]:
# dataset 
dataset = Dataset(path, dev_split, None, 1)

# model 
bundle = _CONFIGS[model_type].bundle
decoder = bundle.get_decoder().cuda()
token_processor = bundle.get_token_processor()
feature_extractor = bundle.get_feature_extractor().cuda()
streaming_feature_extractor = bundle.get_streaming_feature_extractor().cuda()
hop_length = bundle.hop_length
num_samples_segment = bundle.segment_length * hop_length
num_samples_segment_right_context = num_samples_segment + bundle.right_context_length * hop_length

Using custom data configuration default-6e977ecf05190e74
Reusing dataset csv (/home/imgriff/.cache/huggingface/datasets/csv/default-6e977ecf05190e74/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e)


#### Single words recorded at 44.1kHz - resample to 16kHz on the fly

In [9]:
# resampler = T.Resample(44100, 16000, lowpass_filter_width=64,
#                    rolloff=0.9475937167399596, 
#                    resampling_method="kaiser_window",beta=14.769656459379492, dtype=torch.float32)

def load_audio(example):
    wav, sr = torchaudio.load(example['wav_path'])
#     wav = resampler(wav)
    example['audio'] = wav
    example['sampling_rate'] = 16000
    return example


Unpack huggingface dataset from custom dataset object for convenience 

In [10]:
updated_dataset = dataset.dataset.map(load_audio)

Loading cached processed dataset at /home/imgriff/.cache/huggingface/datasets/csv/default-6e977ecf05190e74/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e/cache-c28c661dad041df8.arrow


#### Run Single-Word evaluation

In [11]:
out_file = '../result/emformer_single_word.csv'

with open(out_file,'w',encoding='UTF-8') as f:
                f.write('idx\tbeam\thyp\ttruth\n')

for sample in tqdm(updated_dataset):
    truth = sample["word"].lower()

    waveform = torch.FloatTensor(sample['audio'][0]).cuda()
    # Streaming decode.
#     state, hypothesis = None, None
#     for idx in range(0, len(waveform), num_samples_segment):
#         segment = waveform[idx : idx + num_samples_segment_right_context]
#         segment = torch.nn.functional.pad(segment, (0, num_samples_segment_right_context - len(segment)))
#         with torch.no_grad():
#             features, length = streaming_feature_extractor(segment)
#             hypos, state = decoder.infer(features, length, 10, state=state, hypothesis=hypothesis)
#         hypothesis = hypos[0]
#         transcript = token_processor(hypothesis.tokens, lstrip=False)
#         print(transcript, end="", flush=True)
#     print()

    # Non-streaming decode.
    with torch.no_grad():
        features, length = feature_extractor(waveform)
        hypos = decoder(features, length, 10)
 
            
#     print([token_processor(hypos[ix].tokens) for ix in range(10)])
#     print(f"truth: {sample[1]}")
#     print()

    hypos = [token_processor(hypos[ix].tokens) for ix in range(10)]
    with open(out_file, 'a', encoding='UTF-8') as f:
        for b, hyp in enumerate(hypos):
            f.write('\t'.join([truth, str(b), hyp, truth])+'\n')


  0%|          | 0/200 [00:00<?, ?it/s]

In [12]:
results = pd.read_csv(out_file, sep='\t')

In [13]:
results.head(5)

,idx,beam,hyp,truth
0,the,0,the,the
1,the,1,vidocq,the
2,the,2,the the,the
3,the,3,v,the
4,the,4,va,the


In [14]:
top_match_guesses = results[(results['hyp'] == results['truth']) & (results['beam']==0)]
percent_right = 100 * len(top_match_guesses) / len(results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

166 matched guesses; 83.0% correct


,idx,beam,hyp,truth
0,the,0,the,the
10,and,0,and,and
30,of,0,of,of
40,a,0,a,a
50,that,0,that,that


In [15]:

dmeta = fuzzy.DMetaphone()

results['hyp_dmeta'] = results['hyp'].apply(lambda x: dmeta(x) if isinstance(x, str) else x)
results['truth_dmeta'] = results['truth'].apply(lambda x: dmeta(x))

In [16]:
top_match_guesses = results[(results['hyp_dmeta'] == results['truth_dmeta']) & (results['beam']==0)]
percent_right = 100 * len(top_match_guesses) / len(results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

183 matched guesses; 91.5% correct


,idx,beam,hyp,truth,hyp_dmeta,truth_dmeta
0,the,0,the,the,"[b'0', b'T']","[b'0', b'T']"
10,and,0,and,and,"[b'ANT', None]","[b'ANT', None]"
20,to,0,two,to,"[b'T', None]","[b'T', None]"
30,of,0,of,of,"[b'AF', None]","[b'AF', None]"
40,a,0,a,a,"[b'A', None]","[b'A', None]"


## On noise 0 dBSNR

In [17]:

path = '/om2/user/imgriff/projects/cocktail_party/datasets/GigaSpeech_top_200_words/' # Path to raw LibriSpeech dataset
dev_split = ['GigaSpeech_top_200_words_0SNR']  # Name of data splits to be used as validation set

dataset = Dataset(path, dev_split, None, 1)

updated_dataset = dataset.dataset.map(load_audio)

Using custom data configuration default-c9dd0aeaa110b3ae
Reusing dataset csv (/home/imgriff/.cache/huggingface/datasets/csv/default-c9dd0aeaa110b3ae/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e)
Loading cached processed dataset at /home/imgriff/.cache/huggingface/datasets/csv/default-c9dd0aeaa110b3ae/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e/cache-1aa1c950d7d80a14.arrow


In [18]:
out_file = '../result/emformer_single_word_0dB_SNR.csv'

with open(out_file,'w',encoding='UTF-8') as f:
                f.write('idx\tbeam\thyp\ttruth\n')

for sample in tqdm(updated_dataset):
    truth = sample["word"].lower()

    waveform = torch.FloatTensor(sample['audio'][0]).cuda()
    # Streaming decode.
#     state, hypothesis = None, None
#     for idx in range(0, len(waveform), num_samples_segment):
#         segment = waveform[idx : idx + num_samples_segment_right_context]
#         segment = torch.nn.functional.pad(segment, (0, num_samples_segment_right_context - len(segment)))
#         with torch.no_grad():
#             features, length = streaming_feature_extractor(segment)
#             hypos, state = decoder.infer(features, length, 10, state=state, hypothesis=hypothesis)
#         hypothesis = hypos[0]
#         transcript = token_processor(hypothesis.tokens, lstrip=False)
#         print(transcript, end="", flush=True)
#     print()

    # Non-streaming decode.
    with torch.no_grad():
        features, length = feature_extractor(waveform)
        hypos = decoder(features, length, 10)
 

    hypos = [token_processor(hypos[ix].tokens) for ix in range(10)]
    with open(out_file, 'a', encoding='UTF-8') as f:
        for b, hyp in enumerate(hypos):
            f.write('\t'.join([truth, str(b), hyp, truth])+'\n')


In [19]:
results = pd.read_csv(out_file, sep='\t')

top_match_guesses = results[(results['hyp'] == results['truth']) & (results['beam']==0)]
percent_right = 100 * len(top_match_guesses) / len(results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

45 matched guesses; 22.5% correct


,idx,beam,hyp,truth
70,i,0,i,i
130,so,0,so,so
160,but,0,but,but
270,like,0,like,like
280,what,0,what,what


In [20]:
dmeta = fuzzy.DMetaphone()

results['hyp_dmeta'] = results['hyp'].apply(lambda x: dmeta(x) if isinstance(x, str) else x)
results['truth_dmeta'] = results['truth'].apply(lambda x: dmeta(x))

top_match_guesses = results[(results['hyp_dmeta'] == results['truth_dmeta']) & (results['beam']==0)]
percent_right = 100 * len(top_match_guesses) / len(results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

53 matched guesses; 26.5% correct


,idx,beam,hyp,truth,hyp_dmeta,truth_dmeta
20,to,0,two,to,"[b'T', None]","[b'T', None]"
70,i,0,i,i,"[b'A', None]","[b'A', None]"
130,so,0,so,so,"[b'S', None]","[b'S', None]"
160,but,0,but,but,"[b'PT', None]","[b'PT', None]"
270,like,0,like,like,"[b'LK', None]","[b'LK', None]"


## On noise -5 dBSNR

In [22]:

path = '/om2/user/imgriff/projects/cocktail_party/datasets/GigaSpeech_top_200_words/' # Path to raw LibriSpeech dataset
dev_split = ['GigaSpeech_top_200_words_-5SNR']  # Name of data splits to be used as validation set

dataset = Dataset(path, dev_split, None, 1)

updated_dataset = dataset.dataset.map(load_audio)


Using custom data configuration default-5683877c4c4495f8
Reusing dataset csv (/home/imgriff/.cache/huggingface/datasets/csv/default-5683877c4c4495f8/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e)
Loading cached processed dataset at /home/imgriff/.cache/huggingface/datasets/csv/default-5683877c4c4495f8/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e/cache-c3c5b179e5247f84.arrow


In [23]:
out_file = '../result/emformer_single_word_-5dB_SNR.csv'

with open(out_file,'w',encoding='UTF-8') as f:
                f.write('idx\tbeam\thyp\ttruth\n')

for sample in tqdm(updated_dataset):
    truth = sample["word"].lower()

    waveform = torch.FloatTensor(sample['audio'][0]).cuda()

    # Non-streaming decode.
    with torch.no_grad():
        features, length = feature_extractor(waveform)
        hypos = decoder(features, length, 10)


    hypos = [token_processor(hypos[ix].tokens) for ix in range(10)]
    with open(out_file, 'a', encoding='UTF-8') as f:
        for b, hyp in enumerate(hypos):
            f.write('\t'.join([truth, str(b), hyp, truth])+'\n')


In [29]:
results = pd.read_csv(out_file, sep='\t')

top_match_guesses = results[(results['hyp'] == results['truth']) ]
percent_right = 100 * len(top_match_guesses) / len(results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)


15 matched guesses; 7.5% correct


,idx,beam,hyp,truth
8,the,8,the,the
71,i,1,i,i
101,it,1,it,it
134,so,4,so,so
167,but,7,but,but


In [30]:
dmeta = fuzzy.DMetaphone()

results['hyp_dmeta'] = results['hyp'].apply(lambda x: dmeta(x) if isinstance(x, str) else x)
results['truth_dmeta'] = results['truth'].apply(lambda x: dmeta(x))

top_match_guesses = results[(results['hyp_dmeta'] == results['truth_dmeta'])]
percent_right = 100 * len(top_match_guesses) / len(results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

38 matched guesses; 19.0% correct


,idx,beam,hyp,truth,hyp_dmeta,truth_dmeta
8,the,8,the,the,"[b'0', b'T']","[b'0', b'T']"
47,a,7,i,a,"[b'A', None]","[b'A', None]"
69,in,9,on,in,"[b'AN', None]","[b'AN', None]"
71,i,1,i,i,"[b'A', None]","[b'A', None]"
72,i,2,ah,i,"[b'A', None]","[b'A', None]"
